# LegalIR Task 1: High-Recall Vietnamese Legal Information Retrieval
## UIT Data Science Challenge 2026 — End-to-End Kaggle T4 x2 Production Pipeline

### System Architecture & Objectives:
- **Canonical Legal Structure**: Micro-chunks for statutory precision + Macro-chunks for semantic retrieval.
- **4-Branch Hybrid Candidate Retrieval**: Raw/Legal BM25 + PyVi BM25 + DEk21 Dense Macro + Train-Question Memory + Exact Matcher.
- **Query-Aware Evidence Localization**: Dynamic chunk selection within documents (2–4 chunks/doc).
- **Supervised Cross-Encoder Reranker**: Real LoRA/PEFT fine-tuning on fold-safe hard-negative pairs with duplicate-group blacklist.
- **Learned / OOF Fusion**: Out-of-fold validation with official Codabench scorer equivalence.
- **Strict Invariant Validation & Packaging**: Verification of query completeness, candidate bounds ($1 \le |answer| \le 5$, default 5), duplicate elimination, valid corpus IDs, <4B parameter budget audit, and `submission.zip` packaging containing strictly `submission.json` at root.

### Non-Negotiable Competition Constraints:
1. **Learned Parameter Budget**: Total system parameters strictly `< 4,000,000,000` (4B).
2. **Data Restriction**: Organizer Task 1 data only (no Task 2, no external legal texts, no external LLM APIs).
3. **Hardware Target**: Dual GPU T4 x2 optimized execution (GPU 0: Dense, GPU 1: Reranker).


In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Global Seed & Kaggle Secret / GPU Detection
# ==============================================================================
import os
import sys
import gc
import json
import time
import random
from pathlib import Path
import numpy as np
import torch

# 1. Global reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# 2. Execution mode configuration: "full" for complete competition run, "smoke" for fast verification
RUN_MODE = os.environ.get("LEGALIR_RUN_MODE", "full")  # "full" or "smoke"
print(f"[*] LegalIR Execution Mode: {RUN_MODE.upper()}")

# 3. Secure Kaggle Secret HF_TOKEN retrieval (NEVER print token value)
hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("[+] Hugging Face token securely loaded from Kaggle Secrets.")
except Exception:
    if hf_token:
        print("[+] Hugging Face token present in environment.")
    else:
        print("[-] HF_TOKEN not found in Kaggle Secrets (public models will be used).")

# 4. Hardware and Dual GPU (T4 x2) Detection (P1.10)
device_count = torch.cuda.device_count()
print(f"[+] CUDA Available: {torch.cuda.is_available()} | Device Count: {device_count}")
if device_count > 0:
    for i in range(device_count):
        prop = torch.cuda.get_device_properties(i)
        vram_gb = prop.total_memory / (1024**3)
        print(f"    - GPU {i}: {prop.name} | Total VRAM: {vram_gb:.2f} GB | Compute Capability: {prop.major}.{prop.minor}")
    if device_count >= 2:
        print("[+] Dual GPU environment detected (T4 x2). Multi-GPU staging enabled (GPU 0: Dense, GPU 1: Reranker).")
else:
    print("[!] Running on CPU.")


In [ ]:
# ==============================================================================
# Cell 2: Repository Bootstrap & Minimal Dependencies Installation
# ==============================================================================
import subprocess

# Ensure repo root or /kaggle/working/LegalIR is in sys.path
CWD = Path.cwd()
possible_repo_paths = [
    CWD,
    CWD / "LegalIR",
    Path("/kaggle/working/LegalIR"),
    Path("/kaggle/working"),
]

REPO_ROOT = None
for p in possible_repo_paths:
    if (p / "src" / "pipeline").exists():
        REPO_ROOT = p.resolve()
        break

if REPO_ROOT is None:
    print("[*] Cloning LegalIR repository into /kaggle/working/LegalIR...")
    target_dir = Path("/kaggle/working/LegalIR")
    if not target_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(target_dir)], check=True)
    REPO_ROOT = target_dir.resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Check and install minimal missing dependencies (zero PyTorch reinstallation)
print("[*] Checking and installing minimal required dependencies...")
required_pkgs = []
for mod, pkg in [("bm25s", "bm25s"), ("pyvi", "pyvi"), ("peft", "peft"), ("accelerate", "accelerate")]:
    try:
        __import__(mod)
    except ImportError:
        required_pkgs.append(pkg)

if required_pkgs:
    print(f"[*] Installing missing packages: {required_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location"] + required_pkgs, check=True)
    print("[+] Minimal dependencies installed successfully.")
else:
    print("[+] All required dependencies are already available.")

try:
    commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode("utf-8").strip()
except Exception:
    commit_sha = "unknown"

print(f"[+] Repository Root: {REPO_ROOT}")
print(f"[+] Git Commit SHA : {commit_sha}")
print(f"[+] Python Version  : {sys.version.split()[0]}")
print(f"[+] PyTorch Version : {torch.__version__}")


In [ ]:
# ==============================================================================
# Cell 3: Canonical Dataset & Working Directory Discovery
# ==============================================================================
from src.pipeline.kaggle_train import discover_data_dir, discover_public_test_file

DATA_DIR = discover_data_dir(repo_root=REPO_ROOT)
WORKING_DIR = Path("/kaggle/working/legalir_run") if Path("/kaggle/working").exists() else REPO_ROOT / "artifacts/task1/submissions"
WORKING_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_TEST_FILE = discover_public_test_file(repo_root=REPO_ROOT)

print(f"[+] Canonical Data Directory: {DATA_DIR}")
print(f"[+] Working Directory       : {WORKING_DIR}")
print(f"[+] Public Test Queries     : {PUBLIC_TEST_FILE}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Complete End-to-End Pipeline via run_kaggle_pipeline(...)
# ==============================================================================
from src.pipeline.kaggle_train import run_kaggle_pipeline

print("[*] Launching LegalIR 24-Step Production Pipeline...")
result = run_kaggle_pipeline(
    data_dir=DATA_DIR,
    working_dir=WORKING_DIR,
    run_mode=RUN_MODE,
    hf_token=hf_token,
    public_json_path=PUBLIC_TEST_FILE,
    repo_root=REPO_ROOT,
)

print("\n" + "=" * 80)
print("LEGALIR KAGGLE RUN RESULTS SUMMARY:")
print("=" * 80)
print(f"  - Pipeline Valid          : {result.is_valid}")
print(f"  - Full OOF Mean Recall@5  : {result.cv_report.get('mean_recall@5', 0.0) * 100:.4f}%")
print(f"  - Full OOF Mean Prec@5    : {result.cv_report.get('mean_precision@5', 0.0) * 100:.4f}%")
print(f"  - Candidate Recall@150    : {result.cv_report.get('mean_candidate@150', 0.0) * 100:.4f}%")
print(f"  - Fusion Winner           : {result.fusion_report.get('winning_method', 'N/A')}")
print(f"  - Public Predictions      : {result.public_predictions_count:,} queries")
print(f"  - Learned Parameters      : {result.audit_report.get('total_learned_parameters', 0):,} / 4.0B limit ({result.audit_report.get('budget_utilization_pct', 0.0):.2f}%)")
print(f"  - Total Execution Time    : {result.execution_time_seconds:.2f}s")
print(f"  - Submission JSON         : {result.submission_path}")
print(f"  - Submission ZIP          : {result.submission_zip_path} ({result.submission_zip_path.stat().st_size:,} bytes)")
print(f"  - Submission Manifest     : {result.manifest_path}")
print("=" * 80)
